# Carbon Mapper — plume to analysis-grade raster, quickly

The one workflow most consumers need, first; selection and reference
after. Ground truth: a live-API audit (2026-07).

**The two rules of the current API:**

1. **Plumes are the only door.** Discover via
   `/catalog/plumes/annotated`; every product (L3A crops, the L2B
   parent tile) is reached *from the plume record*. STAC serves history
   only (≤ `v3a`, 2025-12); recent tiles can't be browsed directly.
2. **Versions are per-plume, never hardcoded.** The reader parses each
   plume's `(gas, cmf_type, version)` from its record and derives every
   asset URL from it.


In [ ]:
import os

from georeader.readers.carbonmapper import (
    CarbonMapperConfig,
    CMCollectionSpec,
    CMPlumeImage,
    CMProductFamily,
    CMProductNotSelected,
    api_queries,
)
from georeader.readers.carbonmapper import products as P

# 429-resilient HTTP: Carbon Mapper rate-limits per account. Route every
# call in this kernel through a retry-aware session (honours Retry-After).
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

_session = requests.Session()
_session.mount("https://", HTTPAdapter(max_retries=Retry(
    total=8, backoff_factor=2.0,
    status_forcelist=(429, 500, 502, 503, 504),
    respect_retry_after_header=True,
    allowed_methods=frozenset(["GET", "POST"]),
)))
requests.get, requests.post, requests.request = (
    _session.get, _session.post, _session.request,
)

TOKEN = CarbonMapperConfig.load().refresh_access_token()

# Rasters stream over GDAL/libcurl — pass the token + retries there too.
os.environ["GDAL_HTTP_HEADERS"] = f"Authorization: Bearer {TOKEN}"
os.environ["CPL_VSIL_CURL_ALLOWED_EXTENSIONS"] = ".tif,.TIF"
os.environ["GDAL_HTTP_MAX_RETRY"] = "5"
os.environ["GDAL_HTTP_RETRY_DELAY"] = "3"

# Protagonist: a strong (1954 kg/h) Tanager plume. Its L3A was
# re-versioned to v3d while its L2B parent still serves at v3c — the
# version split the reader resolves automatically.
PLUME_ID = "tan20260331t181625c77s4001-D"
print(f"plume_id = {PLUME_ID}")


## Quickstart — plume → outline → analysis-grade crops

Three objects, three lines each: the typed plume, its product bundle,
its parent tile crops. Everything is lazy until read.


In [ ]:
plume = api_queries.get_plume(TOKEN, PLUME_ID)          # typed record
img = CMPlumeImage.from_cmrawplume(plume, token=TOKEN)  # product bundle
em = f"{plume.emission_auto:.0f} kg/h" if plume.emission_auto is not None else "no emission yet"
print(f"{plume.plume_id}  {em}  spec={plume.collection_spec}")

outline = img.outline          # canonical polygon, EPSG:4326
crop = img.tile_cmf(pad_px=64) # CH4 retrieval @ L2B native resolution
rgb = img.tile_rgb(pad_px=64)  # true-colour context, same grid
print(f"outline bounds: {tuple(round(b, 4) for b in outline.bounds)}")
print(f"tile_cmf: {tuple(crop.values.shape)} @ {crop.crs}")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from pyproj import Transformer
from shapely.ops import transform as shp_transform


def plot_outline(ax, geom_4326, dst_crs):
    """Reproject the plume outline into dst_crs and stroke it in red."""
    if geom_4326 is None:
        return
    tr = Transformer.from_crs("EPSG:4326", dst_crs, always_xy=True)
    g = shp_transform(tr.transform, geom_4326)
    polys = list(g.geoms) if g.geom_type == "MultiPolygon" else [g]
    for p in polys:
        x, y = p.exterior.xy
        ax.plot(x, y, color="red", lw=1.4, solid_capstyle="round")


def to_display(reader, rgb=False):
    """Load a reader -> (array, extent, crs) ready for imshow."""
    geo = reader.load()
    arr = np.asarray(geo.values)
    b = geo.bounds
    extent = (b[0], b[2], b[1], b[3])
    if rgb:
        im = np.moveaxis(arr[:3], 0, -1).astype("float32")
        lo, hi = np.nanpercentile(im, [2, 98])
        return np.clip((im - lo) / max(hi - lo, 1e-9), 0, 1), extent, str(geo.crs)
    a = arr.squeeze().astype("float32")
    nd = reader.nodata
    if nd is not None:
        a = np.where(a == nd, np.nan, a)
    return a, extent, str(geo.crs)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.8), constrained_layout=True)

cmf_arr = np.where(np.asarray(crop.values).squeeze() == -9999, np.nan,
                   np.asarray(crop.values).squeeze()).astype("float32")
b = crop.bounds
im = axes[0].imshow(cmf_arr, extent=(b[0], b[2], b[1], b[3]),
                    origin="upper", cmap="magma")
fig.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04, label="ppm·m")
plot_outline(axes[0], outline, str(crop.crs))
axes[0].set_title("tile_cmf — analysis-grade CH4", fontsize=10)

rgb_im = np.moveaxis(np.asarray(rgb.values)[:3], 0, -1).astype("float32")
lo, hi = np.nanpercentile(rgb_im, [2, 98])
rgb_im = np.clip((rgb_im - lo) / max(hi - lo, 1e-9), 0, 1)
b = rgb.bounds
axes[1].imshow(rgb_im, extent=(b[0], b[2], b[1], b[3]), origin="upper")
plot_outline(axes[1], outline, str(rgb.crs))
axes[1].set_title("tile_rgb — context", fontsize=10)

for ax in axes:
    ax.set_aspect("equal")
fig.suptitle(f"{PLUME_ID}", fontsize=11)
plt.show()


## Selecting products explicitly

Pass `products=` to fetch exactly what you need; access outside the
selection raises instead of returning `None`. The registry
(`products.ALL_PRODUCTS`) is the complete list — including the L3A
thumbnails and PNG quicklooks not shown above.


In [ ]:
lean = CMPlumeImage.from_cmrawplume(
    plume, token=TOKEN, products=(P.PLUME_OUTLINE, P.RGB_TIF),
)
print(f"lean bundle: {sorted(lean.urls)}")
try:
    lean.concentrations
except CMProductNotSelected as exc:
    print(f"unselected access -> {type(exc).__name__}")

print()
print(f"{'product':30s} {'family':8s} description")
print("-" * 88)
for prod in P.ALL_PRODUCTS:
    print(f"{prod.key:30s} {prod.family.value:8s} {prod.description}")


## Sources — site-level context

One call resolves the plume's DBSCAN source; the source detail embeds
every attributed plume for instant per-site stats. Source names get
re-clustered upstream — resolve at run time, never hardcode.


In [ ]:
import pandas as pd

# Plume -> source (None if not yet DBSCAN-clustered), then per-source
# stats from the embedded plume records. Source names are re-clustered
# upstream — resolve them at run time, never hardcode.
src = api_queries.get_source_for_plume(TOKEN, PLUME_ID)
print(f"protagonist source: {src.source_name if src else '(not clustered yet)'}")

permian = api_queries.list_sources(
    TOKEN, bbox=(-104.5, 32.0, -103.5, 32.8), gas="CH4",
)
demo = max(permian, key=lambda s: s.plume_count or 0)
plumes = api_queries.list_plumes_for_source(TOKEN, demo.source_name)
em = pd.Series([p.emission_auto for p in plumes
                if p.emission_auto is not None])
pers = f"{demo.persistence:.2f}" if demo.persistence is not None else "—"
print(f"\n{demo.source_name}: {len(plumes)} detections, persistence={pers}")
print(f"emission kg/h: median={em.median():.0f}  p90={em.quantile(.9):.0f}  "
      f"max={em.max():.0f}")


## Levels at a glance

| Level | Entity | What you get | Keyed by | Latest (2026+) | History (≤ 2025-12) |
|---|---|---|---|---|---|
| Plume record | plume | metadata + URL fields (`plume_tif` → L3A-vis, `con_tif` → L3A-ime) | `plume_id` | ✅ `/catalog/plumes/annotated` — **the only discovery route** | ✅ same |
| L3A vis + ime | image | per-plume crops: mask, concentrations, outlines, rgb, IME set | `plume_id` | ✅ asset proxy, URLs derived from the record | ✅ also in STAC (≤ v3a) |
| L2B + l2b-rgb | tile | full-swath cmf, uncertainty, artifact-mask, uas, rgb | scene name (from `plume_id`) | ✅ asset proxy — reachable **only via a plume**; no browsing (`/catalog/scenes` 401, STAC stale) | ✅ STAC (≤ v3a) |
| Sources (L4) | source | DBSCAN site clusters, stats, embedded plume records | `source_name` (re-clustered — never hardcode) | ✅ REST `/catalog/source*`, live | `l4a-*` STAC ≤ v3a, no per-item assets |
| L2C / L3C | detections / attribution | — | — | not exposed by the reader | — |

Publication timing: plume + L3A appear together (same-day to ~30-day
embargo observed); the L2B parent can lag by days-to-weeks and may sit
one version behind a re-versioned L3A.


## Appendix — product map & access rules

**Per-plume (L3A)** — thumbnail-grade crops, keyed by `plume_id`:
`plume.tif` (mask), `plume-concentrations.tif`,
`ime-cmf-{concentrations,mask}.tif`, `rgb.tif`,
`{plume,ime-cmf}-outline.geojson`, + 5 PNG quicklooks (opt-in).

**Per-scene (L2B)** — full swath, keyed by scene name
(`plume_id.rsplit('-', 1)[0]`): `cmf`, `uncertainty` (+ `-unortho`
variants), `artifact-mask`, `uas.txt`, and `rgb` in the sibling
collection.

| You want | Works | Route |
|---|---|---|
| Discover recent plumes | ✅ | `/catalog/plumes/annotated` (`api_queries.list_plumes`) |
| Latest L3A per-plume products (v3c/v3d) | ✅ | asset-proxy URLs derived from the plume record (`CMPlumeImage`) |
| Latest L2B tile for a known plume/scene | ✅ | asset proxy, plume's version probed first (`img.tile`, `get_image_raster_for_plume`) |
| Anything current via **STAC** | ❌ | registry stops at `-v3a` (items end 2025-12-16) — history only |
| Browse/list recent **tiles** without a plume | ❌ | `/catalog/scenes` 401s; STAC stale — plumes are the only door to current data |
| Plumes attributed to a source | ✅ | embedded records on `/catalog/source/{name}` (`list_plumes_for_source`) |

## See also

- [`api_explore.ipynb`](api_explore.ipynb) — the typed query layer (REST + STAC, filters, exceptions).
- [Carbon Mapper reader API reference](../modules/readers_module.md#carbon-mapper-reader).
